# 01 - Exploratory Data Analysis

Quick look at the master feature table produced by `make features`. This notebook is intentionally short - its purpose is to verify coverage, missingness, and the shape of the headline series before any analysis is run.

Loads `data/processed/features.parquet` if available; falls back to `data/sample/features_sample.parquet` so the notebook also runs on a fresh clone.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
candidates = [ROOT / 'data' / 'processed' / 'features.parquet', ROOT / 'data' / 'sample' / 'features_sample.parquet']
feats_path = next(p for p in candidates if p.exists())
feats = pd.read_parquet(feats_path)
print(f'Loaded {feats_path.relative_to(ROOT)}')
print(f'Shape: {feats.shape}')
feats.head(3)

## Asset coverage and date range

In [ ]:
coverage = feats.groupby('asset').agg(
    first_date=('date', 'min'),
    last_date=('date', 'max'),
    n_days=('date', 'nunique'),
).sort_values('n_days', ascending=False)
coverage

## Missingness across key feature blocks

In [ ]:
block_cols = {
    'price/vol':   ['price', 'volume', 'rolling_30d_volatility', 'drawdown'],
    'sentiment':   ['sentiment_score', 'sentiment_zscore_30d'],
    'liquidity':   ['tvl_usd', 'liquidity_stress_score'],
    'on-chain':    ['tx_count', 'onchain_activity_index'],
    'macro':       ['vix_level', 'sp500_return_7d', 'crypto_macro_corr_30d'],
    'event':       ['event_flag', 'days_since_event'],
}
miss = pd.DataFrame({
    block: feats[[c for c in cols if c in feats.columns]].isna().mean().mean()
    for block, cols in block_cols.items()
}, index=['avg_missing_share']).T
miss

On-chain missingness concentrates in non-BTC / non-ETH assets, which is expected: the platform deliberately does not fabricate on-chain data for assets without a free-tier source.

## Indexed price for the core three assets

In [ ]:
core = feats[feats['asset'].isin(['BTC', 'ETH', 'SOL'])].pivot_table(index='date', columns='asset', values='price')
indexed = core.div(core.iloc[0]).mul(100)
ax = indexed.plot(figsize=(11, 4), title='Indexed price (start = 100)')
ax.set_ylabel('index'); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Realized volatility

In [ ]:
vol = feats[feats['asset'].isin(['BTC', 'ETH', 'SOL'])].pivot_table(index='date', columns='asset', values='rolling_30d_volatility')
ax = vol.plot(figsize=(11, 4), title='30-day realized volatility (annualized)')
ax.set_ylabel('vol'); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Regime distribution

In [ ]:
regime_counts = feats.groupby(['asset', 'regime']).size().unstack(fill_value=0)
regime_counts

Calm and Risk-off are the most populated labels; Momentum and On-chain Activity Spike are intentionally narrow because their rule conditions are strict. The classifier is described in `src/features/regime_classifier.py`.